# Image32 UNet Prototype Recipe

Interactive mirror of `prototyping.runners.image32_unet`.

## 0. Knobs

In [ ]:
RUN_FULL_JOB = False
KNOB_OVERRIDES = {
    "device": "cpu",
    "download": False,
    "n_train": 4,
    "n_eval": 4,
    "batch_cap": 2,
    "eval_samples": 2,
    "n_steps": 20,
    "ckpt_every": 10,
    "sampling_steps": 4,
    "base_channels": 8,
    "channel_mult": (1, 2),
    "attn_resolutions": (8,),
    "num_res_blocks": 1,
    "time_emb_dim": 16,
}

## 1. Imports

In [ ]:
import sys
from pathlib import Path

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src").is_dir() and (candidate / "prototyping").is_dir():
        for path in (candidate, candidate / "src"):
            if str(path) not in sys.path:
                sys.path.insert(0, str(path))
        break

from prototyping.runners import image32_unet as runner
from prototyping.runners.common import artifact_paths, latest_completed_attempt, print_json, count_parameters

## 2. Config Construction

In [ ]:
knobs = runner.default_knobs()
knobs.update(KNOB_OVERRIDES)
config = runner.build_config(knobs)
print_json(runner._config_sections(config))

## 3. Data Bundle

In [ ]:
data = runner.build_data(config)
print_json(runner._summarize_bundle(data))

## 4. Model Construction

In [ ]:
model = runner.build_model(config, knobs["device"])
print_json({
    "model_class": f"{model.__class__.__module__}.{model.__class__.__name__}",
    "backbone_class": f"{model.net.__class__.__module__}.{model.net.__class__.__name__}",
    "parameters": count_parameters(model),
})

## 5. One Forward Loss Update

In [ ]:
update = runner.run_one_update(model, data["train"], config, knobs["device"])
print_json(update)

## 6. Metric Calculation

In [ ]:
metrics = runner.inspect_metrics(config, data, model, knobs["device"])
print_json(metrics)

## 7. Optional Full Job

In [ ]:
run_result = None
if RUN_FULL_JOB:
    run_result = runner.run_full_job(config, knobs["out_dir"], knobs["device"], evaluate=True)
    print_json(run_result)

## 8. Artifact Figure Loading

In [ ]:
attempt_dir = Path(run_result["run_dir"]) if run_result else latest_completed_attempt(knobs["out_dir"])
if attempt_dir is not None:
    print_json(artifact_paths(attempt_dir))